# Setup and Configuration

This section imports the required libraries and defines the main settings.
It sets input paths, filtering rules, sampling targets, and random seeds.


In [ ]:

from __future__ import annotations

import csv
import html
import json
import math
import random
import re
import sqlite3
import statistics
import time
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable

CONFIG = {
    # Input and output
    "reviews_path": "Kindle_Store.jsonl",
    "metadata_path": "meta_Kindle_Store.jsonl",
    "output_dir": "processed_kindle",
    "work_db_name": "preprocess_work.sqlite",

    # Time range, positive interactions, and deduplication
    "year_start": 2014,
    "year_end": 2022,
    "positive_threshold": 4.0,
    "deduplication": "keep_latest_user_item_interaction",
    "require_metadata_match": True,

    # Iterative 5-core filtering
    "min_user_positives": 5,
    "min_item_positives": 5,
    "filter_method": "iterative_5_core",

    # User-level sampling for approximately one million interactions
    "sample_strategy": "user_activity_stratified",
    "target_interactions": 1_000_000,
    "sample_overshoot_ratio": 1.62,

    # Chronological split
    "split": "chronological_leave_one_out",
    "ensure_eval_items_in_train": True,

    # Negative sampling
    "default_num_negatives": 1,
    "negative_exclusion": "all_user_positive_items",
    "negative_unique_per_positive": True,
    "eval_num_negatives": 100,

    # Metadata cleaning
    "missing_verified_purchase": False,
    "remove_categories": ["Kindle Store", "Kindle eBooks"],
    "split_inner_category": True,
    "write_sequence_side_fields": True,

    # Reproducibility and runtime control
    "random_seed": 42,
    "progress_every": 200_000,
    "sqlite_commit_every": 50_000,
    "overwrite": True,
}

REVIEWS_PATH = Path(CONFIG["reviews_path"])
METADATA_PATH = Path(CONFIG["metadata_path"])
OUTPUT_DIR = Path(CONFIG["output_dir"])
WORK_DB_PATH = OUTPUT_DIR / "_work" / CONFIG["work_db_name"]

print("Review data path:", REVIEWS_PATH.resolve())
print("Metadata path:", METADATA_PATH.resolve())
print("Output directory:", OUTPUT_DIR.resolve())


# Utility Functions

These helper functions clean text, parse timestamps, and read JSONL files safely.
They also prepare the temporary SQLite database.


In [ ]:

SMART_QUOTES = str.maketrans({
    "\u2018": "'", "\u2019": "'", "\u201A": "'",
    "\u201C": '"', "\u201D": '"', "\u201E": '"',
    "\u00A0": " ",
})

MARKETING_PHRASES = {
    "BUY NOW", "BESTSELLER", "BEST SELLER", "LIMITED TIME",
    "DESCRIPTION", "TOPICS INCLUDE", "CLICK HERE", "ORDER NOW",
}

def progress(message: str, done: bool = False) -> None:
    end = "\n" if done else "\r"
    print(message.ljust(100), end=end, flush=True)

# Convert valid timestamps to milliseconds.
def normalize_timestamp_ms(value: Any) -> int | None:
    if value is None or value == "":
        return None
    try:
        ts = int(float(value))
    except (TypeError, ValueError, OverflowError):
        return None

    # Support timestamps in seconds and milliseconds
    if ts < 10_000_000_000:
        ts *= 1000

    try:
        datetime.fromtimestamp(ts / 1000, tz=timezone.utc)
    except (ValueError, OSError, OverflowError):
        return None
    return ts

def timestamp_year(ts_ms: int) -> int:
    return datetime.fromtimestamp(ts_ms / 1000, tz=timezone.utc).year

def strip_html(text: str) -> str:
    text = html.unescape(text)
    text = re.sub(r"<[^>]+>", " ", text)
    return text

# Normalize text fields into a clean single-line string.
def normalize_text(text: Any) -> str:
    if text is None:
        return ""
    if isinstance(text, list):
        text = " ".join(str(x) for x in text if x is not None)
    elif isinstance(text, dict):
        text = json.dumps(text, ensure_ascii=False)

    text = str(text).translate(SMART_QUOTES)
    text = strip_html(text)
    text = re.sub(r"[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def clean_title(value: Any) -> str:
    return normalize_text(value)

# Clean and join category labels.
def clean_categories(value: Any) -> str:
    if value is None:
        return ""

    if not isinstance(value, list):
        value = [value]

    remove_set = {x.casefold() for x in CONFIG["remove_categories"]}
    cleaned_parts: list[str] = []

    for raw in value:
        part = normalize_text(raw)
        if not part or part.casefold() in remove_set:
            continue

        if CONFIG["split_inner_category"]:
            # Convert commas and ampersands to semicolons
            subparts = re.split(r"\s*(?:,|&)\s*", part)
            cleaned_parts.extend(x for x in subparts if x)
        else:
            cleaned_parts.append(part)

    # Remove duplicates while preserving order
    seen = set()
    result = []
    for part in cleaned_parts:
        key = part.casefold()
        if key not in seen:
            seen.add(key)
            result.append(part)
    return ";".join(result)

def remove_marketing_noise(text: str) -> str:
    text = re.sub(r"!{2,}", "!", text)
    text = re.sub(r"\*{2,}", " ", text)
    text = re.sub(r"★+", " ", text)

    for phrase in MARKETING_PHRASES:
        text = re.sub(
            rf"(?<!\w){re.escape(phrase)}(?!\w)\s*:?",
            " ",
            text,
            flags=re.IGNORECASE,
        )

    # Separate a period from a following uppercase letter
    text = re.sub(r"(?<=[a-z0-9])\.(?=[A-Z])", ". ", text)
    return re.sub(r"\s+", " ", text).strip()

def clean_features(value: Any) -> str:
    return remove_marketing_noise(normalize_text(value))

def clean_description(value: Any) -> str:
    return normalize_text(value)

def extract_author_name(author: Any) -> str:
    if isinstance(author, dict):
        return normalize_text(author.get("name", ""))
    return normalize_text(author)

def to_bool_int(value: Any) -> int:
    if value is None:
        return int(bool(CONFIG["missing_verified_purchase"]))
    if isinstance(value, bool):
        return int(value)
    if isinstance(value, (int, float)):
        return int(bool(value))
    return int(str(value).strip().lower() in {"1", "true", "yes", "y"})

# Read JSONL records without stopping on malformed lines.
def iter_jsonl(path: Path) -> Iterable[tuple[int, dict[str, Any]]]:
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                yield line_no, {}
                continue
            yield line_no, obj

# Create the temporary SQLite database used by the pipeline.
def connect_db() -> sqlite3.Connection:
    WORK_DB_PATH.parent.mkdir(parents=True, exist_ok=True)

    if CONFIG["overwrite"] and WORK_DB_PATH.exists():
        WORK_DB_PATH.unlink()

    conn = sqlite3.connect(WORK_DB_PATH)
    conn.execute("PRAGMA journal_mode=WAL;")
    conn.execute("PRAGMA synchronous=NORMAL;")
    conn.execute("PRAGMA temp_store=FILE;")
    conn.execute("PRAGMA cache_size=-200000;")  # Approximately 200 MB
    return conn

def create_tables(conn: sqlite3.Connection) -> None:
    conn.executescript("""
    CREATE TABLE IF NOT EXISTS metadata (
        item_id TEXT PRIMARY KEY,
        title TEXT NOT NULL,
        categories TEXT NOT NULL,
        features TEXT NOT NULL,
        description TEXT NOT NULL,
        author TEXT NOT NULL,
        average_rating REAL,
        rating_number INTEGER,
        completeness_score INTEGER NOT NULL
    );

    CREATE TABLE IF NOT EXISTS interactions (
        user_id TEXT NOT NULL,
        item_id TEXT NOT NULL,
        rating REAL NOT NULL,
        timestamp INTEGER NOT NULL,
        verified_purchase INTEGER NOT NULL,
        PRIMARY KEY (user_id, item_id)
    );

    CREATE INDEX IF NOT EXISTS idx_interactions_user
        ON interactions(user_id);

    CREATE INDEX IF NOT EXISTS idx_interactions_item
        ON interactions(item_id);

    CREATE INDEX IF NOT EXISTS idx_interactions_user_time
        ON interactions(user_id, timestamp, item_id);
    """)
    conn.commit()

def count_rows(conn: sqlite3.Connection, table: str) -> int:
    return int(conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0])


# Metadata Processing

This section reads and cleans the item metadata.
For duplicate items, it keeps the record with more complete information.


In [ ]:

# Clean metadata and keep the most complete row for each item.
def ingest_metadata(conn: sqlite3.Connection, stats: dict[str, Any]) -> None:
    if not METADATA_PATH.exists():
        raise FileNotFoundError(f"Metadata file not found: {METADATA_PATH.resolve()}")

    sql = """
    INSERT INTO metadata (
        item_id, title, categories, features, description,
        author, average_rating, rating_number, completeness_score
    )
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    ON CONFLICT(item_id) DO UPDATE SET
        title = excluded.title,
        categories = excluded.categories,
        features = excluded.features,
        description = excluded.description,
        author = excluded.author,
        average_rating = excluded.average_rating,
        rating_number = excluded.rating_number,
        completeness_score = excluded.completeness_score
    WHERE excluded.completeness_score > metadata.completeness_score
    """

    read_lines = 0
    invalid_lines = 0
    accepted_rows = 0
    # Buffer rows to reduce database write overhead.
    batch = []

    for line_no, obj in iter_jsonl(METADATA_PATH):
        read_lines += 1
        if not obj:
            invalid_lines += 1
            continue

        item_id = normalize_text(obj.get("parent_asin"))
        if not item_id:
            invalid_lines += 1
            continue

        title = clean_title(obj.get("title"))
        categories = clean_categories(obj.get("categories"))
        features = clean_features(obj.get("features"))
        description = clean_description(obj.get("description"))
        author = extract_author_name(obj.get("author"))

        try:
            average_rating = (
                float(obj["average_rating"])
                if obj.get("average_rating") not in (None, "")
                else None
            )
        except (TypeError, ValueError):
            average_rating = None

        try:
            rating_number = (
                int(obj["rating_number"])
                if obj.get("rating_number") not in (None, "")
                else None
            )
        except (TypeError, ValueError):
            rating_number = None

        completeness_score = sum(
            bool(x) for x in [title, categories, features, description, author]
        )
        completeness_score += int(average_rating is not None)
        completeness_score += int(rating_number is not None)

        batch.append((
            item_id, title, categories, features, description,
            author, average_rating, rating_number, completeness_score
        ))
        accepted_rows += 1

        if len(batch) >= CONFIG["sqlite_commit_every"]:
            conn.executemany(sql, batch)
            conn.commit()
            batch.clear()

        if read_lines % CONFIG["progress_every"] == 0:
            progress(f"Metadata lines read: {read_lines:,}")

    if batch:
        conn.executemany(sql, batch)
        conn.commit()

    progress(f"Metadata reading completed: {read_lines:,} lines", done=True)

    stats["metadata_lines_read"] = read_lines
    stats["metadata_invalid_lines"] = invalid_lines
    stats["metadata_rows_accepted_before_dedup"] = accepted_rows
    stats["metadata_unique_items"] = count_rows(conn, "metadata")


# Review Processing

This section validates review records and limits them to the selected years.
It keeps the latest interaction for each user-item pair.


In [ ]:

# Load valid reviews and keep the latest record per user-item pair.
def ingest_reviews(conn: sqlite3.Connection, stats: dict[str, Any]) -> None:
    if not REVIEWS_PATH.exists():
        raise FileNotFoundError(f"Review file not found: {REVIEWS_PATH.resolve()}")

    sql = """
    INSERT INTO interactions (
        user_id, item_id, rating, timestamp, verified_purchase
    )
    VALUES (?, ?, ?, ?, ?)
    ON CONFLICT(user_id, item_id) DO UPDATE SET
        rating = excluded.rating,
        timestamp = excluded.timestamp,
        verified_purchase = excluded.verified_purchase
    WHERE excluded.timestamp >= interactions.timestamp
    """

    read_lines = 0
    invalid_lines = 0
    outside_year_range = 0
    accepted_rows = 0
    batch = []

    for line_no, obj in iter_jsonl(REVIEWS_PATH):
        read_lines += 1
        if not obj:
            invalid_lines += 1
            continue

        user_id = normalize_text(obj.get("user_id"))
        item_id = normalize_text(obj.get("parent_asin"))
        ts = normalize_timestamp_ms(obj.get("timestamp"))

        try:
            rating = float(obj.get("rating"))
        except (TypeError, ValueError):
            rating = None

        if not user_id or not item_id or ts is None or rating is None:
            invalid_lines += 1
            continue

        year = timestamp_year(ts)
        if not (CONFIG["year_start"] <= year <= CONFIG["year_end"]):
            outside_year_range += 1
            continue

        verified = to_bool_int(obj.get("verified_purchase"))
        batch.append((user_id, item_id, rating, ts, verified))
        accepted_rows += 1

        if len(batch) >= CONFIG["sqlite_commit_every"]:
            conn.executemany(sql, batch)
            conn.commit()
            batch.clear()

        if read_lines % CONFIG["progress_every"] == 0:
            progress(f"Review lines read: {read_lines:,}")

    if batch:
        conn.executemany(sql, batch)
        conn.commit()

    progress(f"Review reading completed: {read_lines:,} lines", done=True)

    stats["review_lines_read"] = read_lines
    stats["review_invalid_lines"] = invalid_lines
    stats["review_lines_outside_year_range"] = outside_year_range
    stats["review_rows_accepted_before_dedup"] = accepted_rows
    stats["unique_user_item_pairs_after_latest_dedup"] = count_rows(conn, "interactions")


# Positive Filtering and 5-Core

This section keeps positive interactions with matching item metadata.
It repeatedly removes inactive users and rare items until the dataset is stable.


In [ ]:

# Keep positive interactions linked to known items.
def apply_positive_and_metadata_filters(
    conn: sqlite3.Connection,
    stats: dict[str, Any],
) -> None:
    before = count_rows(conn, "interactions")

    conn.execute(
        "DELETE FROM interactions WHERE rating < ?",
        (CONFIG["positive_threshold"],),
    )
    conn.commit()
    after_positive = count_rows(conn, "interactions")

    if CONFIG["require_metadata_match"]:
        conn.execute("""
        DELETE FROM interactions
        WHERE NOT EXISTS (
            SELECT 1
            FROM metadata
            WHERE metadata.item_id = interactions.item_id
        )
        """)
        conn.commit()

    after_metadata = count_rows(conn, "interactions")

    stats["interactions_before_positive_filter"] = before
    stats["positive_interactions_after_rating_filter"] = after_positive
    stats["positive_interactions_after_metadata_match"] = after_metadata

# Repeat user and item filtering until the k-core is stable.
def iterative_k_core(
    conn: sqlite3.Connection,
    min_user: int,
    min_item: int,
    stage_name: str,
    stats: dict[str, Any],
) -> None:
    rounds = 0
    # Store each round for the final preprocessing report.
    history = []

    while True:
        rounds += 1
        before = count_rows(conn, "interactions")

        conn.execute("""
        DELETE FROM interactions
        WHERE user_id IN (
            SELECT user_id
            FROM interactions
            GROUP BY user_id
            HAVING COUNT(*) < ?
        )
        """, (min_user,))
        conn.commit()

        after_user = count_rows(conn, "interactions")

        conn.execute("""
        DELETE FROM interactions
        WHERE item_id IN (
            SELECT item_id
            FROM interactions
            GROUP BY item_id
            HAVING COUNT(*) < ?
        )
        """, (min_item,))
        conn.commit()

        after_item = count_rows(conn, "interactions")
        history.append({
            "round": rounds,
            "before": before,
            "after_user_filter": after_user,
            "after_item_filter": after_item,
        })

        print(
            f"{stage_name} round {rounds}: "
            f"{before:,} -> {after_user:,} -> {after_item:,}"
        )

        if after_item == before:
            break
        if after_item == 0:
            raise RuntimeError(
                f"No data remains after {stage_name}. Check the year range, positive threshold, or 5-core thresholds."
            )

    stats[f"{stage_name}_rounds"] = rounds
    stats[f"{stage_name}_history"] = history
    stats[f"{stage_name}_final_interactions"] = count_rows(conn, "interactions")
    stats[f"{stage_name}_final_users"] = int(
        conn.execute("SELECT COUNT(DISTINCT user_id) FROM interactions").fetchone()[0]
    )
    stats[f"{stage_name}_final_items"] = int(
        conn.execute("SELECT COUNT(DISTINCT item_id) FROM interactions").fetchone()[0]
    )


# User-Level Sampling

This section reduces the dataset when it is larger than the target size.
Users are grouped by activity level, and complete user histories are sampled.


In [ ]:

def activity_bin(n: int) -> int:
    return int(math.log2(max(n, 1)))

# Sample whole users while preserving different activity levels.
def sample_users_if_needed(
    conn: sqlite3.Connection,
    stats: dict[str, Any],
) -> None:
    target = int(CONFIG["target_interactions"])
    overshoot_target = int(target * float(CONFIG["sample_overshoot_ratio"]))
    total = count_rows(conn, "interactions")

    stats["interactions_before_user_sampling"] = total
    stats["sampling_target_interactions"] = target
    stats["sampling_overshoot_target"] = overshoot_target

    if total <= target:
        print(f"Only {total:,} interactions are available. Sampling is skipped.")
        stats["sampling_applied"] = False
        return

    rows = conn.execute("""
        SELECT user_id, COUNT(*) AS n
        FROM interactions
        GROUP BY user_id
    """).fetchall()

    rng = random.Random(CONFIG["random_seed"])
    # Group users by log-scaled interaction counts.
    buckets: dict[int, list[tuple[str, int]]] = defaultdict(list)

    for user_id, n in rows:
        buckets[activity_bin(int(n))].append((user_id, int(n)))

    bucket_totals = {
        key: sum(n for _, n in users)
        for key, users in buckets.items()
    }

    selected: set[str] = set()
    remaining: list[tuple[str, int]] = []
    selected_interactions = 0

    # Allocate interaction quotas proportionally across activity groups
    for key in sorted(buckets):
        users = buckets[key]
        rng.shuffle(users)
        quota = overshoot_target * bucket_totals[key] / total
        bucket_selected = 0

        for user_id, n in users:
            new_value = bucket_selected + n
            if new_value <= quota or abs(new_value - quota) < abs(bucket_selected - quota):
                selected.add(user_id)
                selected_interactions += n
                bucket_selected = new_value
            else:
                remaining.append((user_id, n))

    # Use remaining users to move closer to the target
    rng.shuffle(remaining)
    for user_id, n in remaining:
        if selected_interactions >= overshoot_target:
            break
        if abs((selected_interactions + n) - overshoot_target) <= abs(
            selected_interactions - overshoot_target
        ):
            selected.add(user_id)
            selected_interactions += n

    conn.execute("DROP TABLE IF EXISTS selected_users")
    conn.execute("CREATE TEMP TABLE selected_users (user_id TEXT PRIMARY KEY)")
    conn.executemany(
        "INSERT INTO selected_users(user_id) VALUES (?)",
        ((u,) for u in selected),
    )
    conn.commit()

    conn.execute("""
        DELETE FROM interactions
        WHERE user_id NOT IN (SELECT user_id FROM selected_users)
    """)
    conn.commit()

    after = count_rows(conn, "interactions")
    stats["sampling_applied"] = True
    stats["sampled_users_before_second_core"] = len(selected)
    stats["sampled_interactions_before_second_core"] = after

    print(
        f"User-level stratified sampling completed: {total:,} -> {after:,} interactions, "
        f"{len(selected):,} users retained."
    )


# Chronological Data Split

This section orders each user's interactions by time.
The last two items become validation and test records, while earlier items form the training sequence.
It also checks that evaluation items appear in the training data.


In [ ]:

Interaction = tuple[str, int, float, int]
UserSequence = dict[str, Any]

# Build chronological interaction sequences for each user.
def load_user_sequences(
    conn: sqlite3.Connection,
    stats: dict[str, Any],
) -> list[UserSequence]:
    cursor = conn.execute("""
        SELECT user_id, item_id, timestamp, rating, verified_purchase
        FROM interactions
        ORDER BY user_id, timestamp, item_id
    """)

    sequences: list[UserSequence] = []
    current_user: str | None = None
    current_interactions: list[Interaction] = []

    # Save the current user after all of their rows are read.
    def flush() -> None:
        nonlocal current_user, current_interactions
        if current_user is None:
            return
        if len(current_interactions) >= CONFIG["min_user_positives"]:
            sequences.append({
                "user_id": current_user,
                "interactions": current_interactions,
            })
        current_user = None
        current_interactions = []

    for user_id, item_id, ts, rating, verified in cursor:
        if current_user is None:
            current_user = user_id
        elif user_id != current_user:
            flush()
            current_user = user_id

        current_interactions.append((
            item_id,
            int(ts),
            float(rating),
            int(verified),
        ))

    flush()

    stats["users_loaded_for_split"] = len(sequences)
    stats["interactions_loaded_for_split"] = sum(
        len(x["interactions"]) for x in sequences
    )
    return sequences

# Remove users whose validation or test item is unseen in training.
def ensure_eval_items_seen_in_train(
    sequences: list[UserSequence],
    stats: dict[str, Any],
) -> list[UserSequence]:
    if not CONFIG["ensure_eval_items_in_train"]:
        stats["eval_train_coverage_filter_applied"] = False
        return sequences

    stats["eval_train_coverage_filter_applied"] = True
    rounds = 0
    removed_total = 0
    current = sequences

    while True:
        rounds += 1
        train_items = {
            item_id
            for user in current
            for item_id, _, _, _ in user["interactions"][:-2]
        }

        kept = []
        removed = 0

        for user in current:
            interactions = user["interactions"]
            val_item = interactions[-2][0]
            test_item = interactions[-1][0]

            if val_item in train_items and test_item in train_items:
                kept.append(user)
            else:
                removed += 1

        removed_total += removed
        print(
            f"Training-item coverage check round {rounds}: "
            f"{len(kept):,} users retained and {removed:,} users removed."
        )

        current = kept
        if removed == 0:
            break
        if not current:
            raise RuntimeError("No users remain after the training-item coverage check.")

    stats["eval_train_coverage_rounds"] = rounds
    stats["users_removed_for_unseen_eval_items"] = removed_total
    stats["users_after_eval_train_coverage"] = len(current)
    return current


# Output File Generation

This section creates index mappings and writes the processed CSV and JSON files.
It also generates negative samples and summarizes the final dataset.


In [ ]:

def write_json(path: Path, obj: Any) -> None:
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

# Draw items that the user has not interacted with.
def sample_negative_items(
    catalog_size: int,
    positive_items: set[int],
    k: int,
    rng: random.Random,
) -> list[int]:
    available_count = catalog_size - len(positive_items)
    if available_count <= 0 or k <= 0:
        return []

    if CONFIG["negative_unique_per_positive"]:
        k = min(k, available_count)

    result: list[int] = []
    used: set[int] = set()
    # Limit random retries for users with many positive items.
    max_attempts = max(100, k * 30)

    attempts = 0
    while len(result) < k and attempts < max_attempts:
        candidate = rng.randrange(catalog_size)
        attempts += 1

        if candidate in positive_items:
            continue
        if CONFIG["negative_unique_per_positive"] and candidate in used:
            continue

        result.append(candidate)
        used.add(candidate)

    if len(result) < k:
        # Fallback for extremely dense users
        remaining = [
            i for i in range(catalog_size)
            if i not in positive_items and i not in used
        ]
        rng.shuffle(remaining)
        result.extend(remaining[: k - len(result)])

    return result

# Assign stable integer indices to users and items.
def build_mappings(
    sequences: list[UserSequence],
) -> tuple[dict[str, int], dict[str, int]]:
    user_ids = sorted(user["user_id"] for user in sequences)
    item_ids = sorted({
        item_id
        for user in sequences
        for item_id, _, _, _ in user["interactions"]
    })

    user2idx = {user_id: idx for idx, user_id in enumerate(user_ids)}
    item2idx = {item_id: idx for idx, item_id in enumerate(item_ids)}
    return user2idx, item2idx

# Write splits, sequences, mappings, and training triplets.
def write_core_outputs(
    conn: sqlite3.Connection,
    sequences: list[UserSequence],
    user2idx: dict[str, int],
    item2idx: dict[str, int],
    stats: dict[str, Any],
) -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    idx2user = {str(idx): user_id for user_id, idx in user2idx.items()}
    idx2item = {str(idx): item_id for item_id, idx in item2idx.items()}

    write_json(OUTPUT_DIR / "user2idx.json", user2idx)
    write_json(OUTPUT_DIR / "item2idx.json", item2idx)
    write_json(OUTPUT_DIR / "idx2user.json", idx2user)
    write_json(OUTPUT_DIR / "idx2item.json", idx2item)

    interactions_path = OUTPUT_DIR / "interactions_clean.csv"
    sequence_path = OUTPUT_DIR / "train_sequences.csv"
    triplet_path = OUTPUT_DIR / "train_triplets.csv"
    val_path = OUTPUT_DIR / "val.csv"
    test_path = OUTPUT_DIR / "test.csv"

    rng = random.Random(CONFIG["random_seed"])
    catalog_size = len(item2idx)

    train_interaction_count = 0
    triplet_count = 0

    with (
        interactions_path.open("w", newline="", encoding="utf-8") as f_interactions,
        sequence_path.open("w", newline="", encoding="utf-8") as f_sequences,
        triplet_path.open("w", newline="", encoding="utf-8") as f_triplets,
        val_path.open("w", newline="", encoding="utf-8") as f_val,
        test_path.open("w", newline="", encoding="utf-8") as f_test,
    ):
        interactions_writer = csv.writer(f_interactions)
        sequences_writer = csv.writer(f_sequences)
        triplets_writer = csv.writer(f_triplets)
        val_writer = csv.writer(f_val)
        test_writer = csv.writer(f_test)

        interactions_writer.writerow([
            "user_idx", "item_idx", "user_id", "parent_asin",
            "rating", "timestamp", "verified_purchase", "split",
        ])

        if CONFIG["write_sequence_side_fields"]:
            sequences_writer.writerow([
                "user_idx", "item_seq", "timestamp_seq",
                "rating_seq", "verified_seq",
            ])
        else:
            sequences_writer.writerow(["user_idx", "item_seq"])

        triplets_writer.writerow([
            "user_idx", "pos_idx", "neg_idx",
            "pos_timestamp", "pos_rating", "pos_verified_purchase",
        ])
        val_writer.writerow(["user_idx", "pos_idx"])
        test_writer.writerow(["user_idx", "pos_idx"])

        for user in sequences:
            user_id = user["user_id"]
            user_idx = user2idx[user_id]
            interactions: list[Interaction] = user["interactions"]

            # Reserve the last two interactions for validation and testing.
            train = interactions[:-2]
            val = interactions[-2]
            test = interactions[-1]

            all_positive_indices = {
                item2idx[item_id]
                for item_id, _, _, _ in interactions
            }

            train_item_indices = [
                item2idx[item_id]
                for item_id, _, _, _ in train
            ]
            train_timestamps = [str(ts) for _, ts, _, _ in train]
            train_ratings = [str(rating) for _, _, rating, _ in train]
            train_verified = [str(verified) for _, _, _, verified in train]

            if CONFIG["write_sequence_side_fields"]:
                sequences_writer.writerow([
                    user_idx,
                    " ".join(map(str, train_item_indices)),
                    " ".join(train_timestamps),
                    " ".join(train_ratings),
                    " ".join(train_verified),
                ])
            else:
                sequences_writer.writerow([
                    user_idx,
                    " ".join(map(str, train_item_indices)),
                ])

            val_writer.writerow([user_idx, item2idx[val[0]]])
            test_writer.writerow([user_idx, item2idx[test[0]]])

            for split_name, split_rows in (
                ("train", train),
                ("validation", [val]),
                ("test", [test]),
            ):
                for item_id, ts, rating, verified in split_rows:
                    interactions_writer.writerow([
                        user_idx,
                        item2idx[item_id],
                        user_id,
                        item_id,
                        rating,
                        ts,
                        bool(verified),
                        split_name,
                    ])

            for item_id, ts, rating, verified in train:
                pos_idx = item2idx[item_id]
                negatives = sample_negative_items(
                    catalog_size=catalog_size,
                    positive_items=all_positive_indices,
                    k=int(CONFIG["default_num_negatives"]),
                    rng=rng,
                )

                for neg_idx in negatives:
                    triplets_writer.writerow([
                        user_idx,
                        pos_idx,
                        neg_idx,
                        ts,
                        rating,
                        bool(verified),
                    ])
                    triplet_count += 1

                train_interaction_count += 1

    stats["num_users"] = len(user2idx)
    stats["num_items"] = len(item2idx)
    stats["num_interactions"] = sum(
        len(user["interactions"]) for user in sequences
    )
    stats["num_train_interactions"] = train_interaction_count
    stats["num_validation_interactions"] = len(sequences)
    stats["num_test_interactions"] = len(sequences)
    stats["num_train_triplets"] = triplet_count

# Export metadata only for items in the final dataset.
def write_metadata_outputs(
    conn: sqlite3.Connection,
    item2idx: dict[str, int],
    stats: dict[str, Any],
) -> None:
    clean_path = OUTPUT_DIR / "items_metadata_clean.csv"
    optional_path = OUTPUT_DIR / "item_metadata.csv"

    final_items = set(item2idx)

    metadata_rows = {}
    for row in conn.execute("""
        SELECT
            item_id, title, categories, features, description,
            author, average_rating, rating_number
        FROM metadata
    """):
        if row[0] in final_items:
            metadata_rows[row[0]] = row[1:]

    with (
        clean_path.open("w", newline="", encoding="utf-8") as f_clean,
        optional_path.open("w", newline="", encoding="utf-8") as f_optional,
    ):
        clean_writer = csv.writer(f_clean)
        optional_writer = csv.writer(f_optional)

        clean_writer.writerow([
            "item_idx", "parent_asin", "title",
            "categories", "features", "description",
        ])
        optional_writer.writerow([
            "item_idx", "title", "categories", "author",
            "average_rating", "rating_number",
        ])

        for item_id, item_idx in sorted(
            item2idx.items(),
            key=lambda x: x[1],
        ):
            title, categories, features, description, author, avg, n = (
                metadata_rows.get(item_id, ("", "", "", "", "", None, None))
            )

            clean_writer.writerow([
                item_idx, item_id, title,
                categories, features, description,
            ])
            optional_writer.writerow([
                item_idx, title, categories, author, avg, n,
            ])

    stats["metadata_rows_written"] = len(item2idx)

# Create fixed negative candidates for validation and testing.
def write_eval_negatives_if_requested(
    sequences: list[UserSequence],
    user2idx: dict[str, int],
    item2idx: dict[str, int],
    stats: dict[str, Any],
) -> None:
    k = int(CONFIG["eval_num_negatives"])
    if k <= 0:
        stats["eval_negative_files_written"] = False
        return

    rng = random.Random(CONFIG["random_seed"] + 1)
    catalog_size = len(item2idx)

    for split_name in ("val", "test"):
        path = OUTPUT_DIR / f"{split_name}_negatives.csv"
        with path.open("w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(["user_idx", "neg_items"])

            for user in sequences:
                user_idx = user2idx[user["user_id"]]
                positives = {
                    item2idx[item_id]
                    for item_id, _, _, _ in user["interactions"]
                }
                negatives = sample_negative_items(
                    catalog_size,
                    positives,
                    k,
                    rng,
                )
                writer.writerow([
                    user_idx,
                    " ".join(map(str, negatives)),
                ])

    stats["eval_negative_files_written"] = True
    stats["eval_num_negatives_per_user"] = k

# Summarize the final dataset and sequence lengths.
def finalize_stats(
    sequences: list[UserSequence],
    stats: dict[str, Any],
) -> None:
    lengths = [len(user["interactions"]) for user in sequences]
    train_lengths = [x - 2 for x in lengths]

    stats["positive_threshold"] = CONFIG["positive_threshold"]
    stats["min_user_positives"] = CONFIG["min_user_positives"]
    stats["min_item_positives"] = CONFIG["min_item_positives"]
    stats["split"] = CONFIG["split"]
    stats["sample_strategy"] = CONFIG["sample_strategy"]
    stats["random_seed"] = CONFIG["random_seed"]

    stats["avg_full_sequence_length"] = (
        sum(lengths) / len(lengths) if lengths else 0
    )
    stats["median_full_sequence_length"] = (
        statistics.median(lengths) if lengths else 0
    )
    stats["min_full_sequence_length"] = min(lengths) if lengths else 0
    stats["max_full_sequence_length"] = max(lengths) if lengths else 0

    stats["avg_train_sequence_length"] = (
        sum(train_lengths) / len(train_lengths) if train_lengths else 0
    )
    stats["median_train_sequence_length"] = (
        statistics.median(train_lengths) if train_lengths else 0
    )


# Run the Preprocessing Pipeline

This section runs all preprocessing stages in the correct order.
It records the total runtime and saves the configuration and statistics.


In [ ]:

# Run every preprocessing stage in order.
def run_preprocessing() -> dict[str, Any]:
    started = time.time()
    stats: dict[str, Any] = {}

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    conn = connect_db()

    try:
        create_tables(conn)

        print("\n[1/9] Reading item metadata")
        ingest_metadata(conn, stats)

        print("\n[2/9] Reading reviews and retaining the latest user-item rating")
        ingest_reviews(conn, stats)

        print("\n[3/9] Applying positive filtering and metadata matching")
        apply_positive_and_metadata_filters(conn, stats)

        print("\n[4/9] Running the first iterative 5-core filter")
        iterative_k_core(
            conn,
            min_user=CONFIG["min_user_positives"],
            min_item=CONFIG["min_item_positives"],
            stage_name="first_5_core",
            stats=stats,
        )

        print("\n[5/9] Running user-level stratified sampling")
        sample_users_if_needed(conn, stats)

        print("\n[6/9] Running the second iterative 5-core filter")
        iterative_k_core(
            conn,
            min_user=CONFIG["min_user_positives"],
            min_item=CONFIG["min_item_positives"],
            stage_name="second_5_core",
            stats=stats,
        )

        print("\n[7/9] Loading chronological sequences and checking evaluation items")
        sequences = load_user_sequences(conn, stats)
        sequences = ensure_eval_items_seen_in_train(sequences, stats)

        print("\n[8/9] Generating mappings, model files, and metadata")
        user2idx, item2idx = build_mappings(sequences)
        write_core_outputs(
            conn, sequences, user2idx, item2idx, stats
        )
        write_metadata_outputs(conn, item2idx, stats)
        write_eval_negatives_if_requested(
            sequences, user2idx, item2idx, stats
        )

        print("\n[9/9] Writing configuration and statistics")
        finalize_stats(sequences, stats)
        stats["elapsed_seconds"] = round(time.time() - started, 2)

        write_json(OUTPUT_DIR / "preprocess_config.json", CONFIG)
        write_json(OUTPUT_DIR / "preprocess_stats.json", stats)

        print("\nData preprocessing completed.")
        print("Output directory:", OUTPUT_DIR.resolve())
        print(
            f"Users: {stats['num_users']:,}, "
            f"items: {stats['num_items']:,}, "
            f"positive interactions: {stats['num_interactions']:,}"
        )
        print(
            f"Training interactions: {stats['num_train_interactions']:,}, "
            f"validation interactions: {stats['num_validation_interactions']:,}, "
            f"test interactions: {stats['num_test_interactions']:,}"
        )
        return stats

    finally:
        conn.close()

# Uncomment the next line to run the full pipeline:
stats = run_preprocessing()
